In [1]:
import pandas as pd

In [2]:
df= pd.read_csv("./data/ooo.csv")

In [4]:
df.head()

,DOT_NUMBER,LEGAL_NAME,DBA_NAME,OOS_DATE,OOS_REASON,STATUS,RESCIND_DATE
0,1438,AUSTIN URETHANE INC,NaN,2022-07-09,Unsatisfactory = Unfit,ACTIVE,2022-07-11
1,6050,S M TRANSPORT INC,NaN,2007-07-11,90 day failure to pay fine,INACTIVE,NaN
2,7660,NEW CHURCH FARMERS SUPPLY INC,NaN,2010-05-24,90 day failure to pay fine,INACTIVE,NaN
3,11891,H DAVID PITZER TRUCKING INC,NaN,2018-04-23,90 day failure to pay fine,INACTIVE,NaN
4,13172,LARRY TRAPP TRUCKING INC,NaN,2022-03-16,Unsatisfactory = Unfit,ACTIVE,2022-03-31


In [12]:
df.shape

(396307, 7)

In [13]:
total_unique = df['DOT_NUMBER'].nunique()
print(total_unique)

341354


In [14]:
import pandas as pd

# Top 100 most frequent DOT numbers
top100 = (
    df['DOT_NUMBER']
    .value_counts()
    .head(100)
    .reset_index()
)

# Rename columns
top100.columns = ['DOT_NUMBER', 'COUNT']

# Display
print(top100)

# Save to CSV
top100.to_csv('top100_dot_numbers.csv', index=False)

    DOT_NUMBER  COUNT
0      2070351     14
1      2817490     12
2      1063779     11
3      2037384     11
4      3048225     11
..         ...    ...
95     2956245      7
96     2963209      7
97     2981232      7
98     2990270      7
99     2991327      7

[100 rows x 2 columns]


In [ ]:
import pandas as pd
from sodapy import Socrata
import time

# -----------------------------
# Existing function
# -----------------------------
def get_carrier_details(dot_number: int) -> dict:
    try:
        client = Socrata("data.transportation.gov", None)
        dataset_id = "az4n-8mr2"

        where_clause = f"dot_number = {dot_number}"
        results = client.get(dataset_id, where=where_clause, limit=1)

        if not results:
            return {
                "dot_number": dot_number,
                "error": f"No carrier found with DOT number {dot_number}",
                "success": False
            }

        results_df = pd.DataFrame.from_records(results)

        return {
            "dot_number": dot_number,
            "data": results,
            "dataframe": results_df,
            "success": True
        }

    except Exception as e:
        return {
            "dot_number": dot_number,
            "error": str(e),
            "success": False
        }

# -----------------------------
# Read the top-100 DOT file
# -----------------------------
top100 = pd.read_csv("top100_dot_numbers.csv")

# -----------------------------
# Fetch carrier details
# -----------------------------
enriched_rows = []

for dot in top100["DOT_NUMBER"]:

    result = get_carrier_details(int(dot))

    if result.get("success"):

        row = result["dataframe"].iloc[0].to_dict()

        # Keep useful fields only if present
        enriched = {
            "DOT_NUMBER": row.get("dot_number"),
            "LEGAL_NAME": row.get("legal_name"),
            "DBA_NAME": row.get("dba_name"),
            "PHY_STREET": row.get("phy_street"),
            "PHY_CITY": row.get("phy_city"),
            "PHY_STATE": row.get("phy_state"),
            "PHY_ZIP": row.get("phy_zip"),
            "MAILING_STREET": row.get("mailing_street"),
            "MAILING_CITY": row.get("mailing_city"),
            "MAILING_STATE": row.get("mailing_state"),
            "MAILING_ZIP": row.get("mailing_zip"),
            "TELEPHONE": row.get("telephone"),
            "FAX": row.get("fax"),
            "EMAIL": row.get("email_address"),  # may be missing
            "CARRIER_OPERATION": row.get("carrier_operation"),
            "CARRIER_OPERATION_DESC": row.get("carrier_operation_desc"),
            "ENTITY_TYPE": row.get("entity_type"),
            "OPERATING_STATUS": row.get("operating_status"),
            "OPERATING_STATUS_DESC": row.get("operating_status_desc"),
            "USDOT_STATUS": row.get("usdot_status"), 
            "MCS150_DATE": row.get("mcs150_date"),
            "DRIVER_TOTAL": row.get("driver_total"),
            "VEHICLE_TOTAL": row.get("vehicle_total"),
            "OUT_OF_SERVICE_DATE": row.get("oos_date"),
            "SAFETY_RATING": row.get("safety_rating")
        }

        enriched_rows.append(enriched)

    else:
        enriched_rows.append({
            "DOT_NUMBER": dot,
            "ERROR": result.get("error")
        })

    # polite delay to avoid throttling
    time.sleep(2)

# -----------------------------
# Create final enriched table
# -----------------------------
enriched_df = pd.DataFrame(enriched_rows)



ValueError: You are trying to merge on int64 and str columns for key 'DOT_NUMBER'. If you wish to proceed you should use pd.concat

In [17]:
# Ensure both DOT_NUMBER columns are numeric
top100['DOT_NUMBER'] = pd.to_numeric(top100['DOT_NUMBER'], errors='coerce')
enriched_df['DOT_NUMBER'] = pd.to_numeric(enriched_df['DOT_NUMBER'], errors='coerce')

# Optional: drop rows where conversion failed
top100 = top100.dropna(subset=['DOT_NUMBER'])
enriched_df = enriched_df.dropna(subset=['DOT_NUMBER'])

# Convert to integer type
top100['DOT_NUMBER'] = top100['DOT_NUMBER'].astype('int64')
enriched_df['DOT_NUMBER'] = enriched_df['DOT_NUMBER'].astype('int64')

# Now merge
final_df = top100.merge(enriched_df, on='DOT_NUMBER', how='left')

In [19]:
final_df.to_csv("top100_carriers_enriched.csv", index=False)

In [21]:
final_df.columns

Index(['DOT_NUMBER', 'COUNT', 'LEGAL_NAME', 'DBA_NAME', 'PHY_STREET',
       'PHY_CITY', 'PHY_STATE', 'PHY_ZIP', 'MAILING_STREET', 'MAILING_CITY',
       'MAILING_STATE', 'MAILING_ZIP', 'TELEPHONE', 'FAX', 'EMAIL',
       'CARRIER_OPERATION', 'CARRIER_OPERATION_DESC', 'ENTITY_TYPE',
       'OPERATING_STATUS', 'OPERATING_STATUS_DESC', 'USDOT_STATUS',
       'MCS150_DATE', 'DRIVER_TOTAL', 'VEHICLE_TOTAL', 'OUT_OF_SERVICE_DATE',
       'SAFETY_RATING'],
      dtype='str')

In [30]:
df=final_df

In [31]:
print('Unique states:', df['PHY_STATE'].nunique())
print('Unique zips:', df['PHY_ZIP'].astype(str).nunique())

print('\nTop state codes:')
print(df['PHY_STATE'].value_counts().head(20))

print('\nSample unusual state codes:')
print(sorted(df['PHY_STATE'].dropna().unique())[:50])

Unique states: 28
Unique zips: 92

Top state codes:
PHY_STATE
TX    15
NC    14
GA    14
CA     7
NJ     5
MA     4
SO     4
MS     4
PA     4
MO     3
TN     3
CT     3
FL     3
SI     2
NY     2
WA     1
CI     1
SC     1
RI     1
HI     1
Name: count, dtype: int64

Sample unusual state codes:
['CA', 'CI', 'CO', 'CT', 'DF', 'FL', 'GA', 'HI', 'LA', 'MA', 'MD', 'MO', 'MS', 'NC', 'NJ', 'NY', 'OH', 'OK', 'PA', 'QC', 'RI', 'SC', 'SI', 'SO', 'TA', 'TN', 'TX', 'WA']


In [ ]:
import pandas as pd
import requests
from datetime import date

BASE_URL = "https://data.transportation.gov/resource/p2mt-9ige.csv"

def get_daily_oos(date_str=None, limit=1000, offset=0):
    """
    Download up to `limit` OOS inspection records for a specific date.

    Parameters
    ----------
    date_str : str
        Date in YYYY-MM-DD format. Defaults to today.
    limit : int
        Number of rows to fetch.
    offset : int
        Pagination offset.

    Returns
    -------
    pandas.DataFrame
    """

    if date_str is None:
        date_str = date.today().isoformat()

    params = {
        "$limit": limit,
        "$offset": offset,
        "$order": "inspection_date DESC",
        # Keep only OOS inspections for that day
        "$where": (
            f"inspection_date between '{date_str}T00:00:00' "
            f"and '{date_str}T23:59:59' "
            f"AND oos_indicator = 'Y'"
        )
    }

    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()

    from io import StringIO
    return pd.read_csv(StringIO(r.text))

# Example: fetch 1,000 OOS records for a specific day
target_date = "2026-08-14"

df = get_daily_oos(target_date, limit=1000)

print(f"Rows downloaded: {len(df)}")
print(df.head())

# Save to CSV
output_file = f"oos_{target_date}.csv"
df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")

In [ ]:
import pandas as pd
from datetime import datetime

url = "https://data.transportation.gov/resource/p2mt-9ige.csv"

# Load the latest CSV directly
df = pd.read_csv(url)

print("Total rows downloaded:", len(df))
print(df.head())

# Keep only OOS records if the column exists
if 'oos_indicator' in df.columns:
    df = df[df['oos_indicator'] == 'Y']

# Convert inspection date to datetime if present
if 'inspection_date' in df.columns:
    df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
    df = df.sort_values('inspection_date', ascending=False)

# Take the latest 1000 records
latest_1000 = df.head(1000).copy()

print("Latest OOS rows:", len(latest_1000))

# Save with today's date
today = datetime.today().strftime('%Y-%m-%d')
output_file = f"latest_oos_{today}.csv"

latest_1000.to_csv(output_file, index=False)

print(f"Saved: {output_file}")

In [5]:
import os
import pandas as pd
from get_carrier_details import *

DAILY_OOS_FILE = "./daily/daily_ooo.csv"
CARRIER_DETAILS_FILE = "carrier_details.csv"


def update_carrier_details():
    # ---------------------------------------------------------
    # 1. Read daily OOS data
    # ---------------------------------------------------------
    daily_ooo = pd.read_csv(DAILY_OOS_FILE, dtype={"DOT_NUMBER": str})

    daily_ooo["DOT_NUMBER"] = (
        daily_ooo["DOT_NUMBER"]
        .astype(str)
        .str.strip()
    )

    # Remove empty / invalid DOT numbers
    daily_ooo = daily_ooo[
        daily_ooo["DOT_NUMBER"].notna()
        & (daily_ooo["DOT_NUMBER"] != "")
        & (daily_ooo["DOT_NUMBER"].str.lower() != "nan")
    ].copy()

    daily_dots = set(daily_ooo["DOT_NUMBER"].unique())

    print(f"DOT numbers in daily OOS: {len(daily_dots)}")

    # ---------------------------------------------------------
    # 2. Read existing carrier details
    # ---------------------------------------------------------
    if os.path.exists(CARRIER_DETAILS_FILE):
        carrier_details = pd.read_csv(
            CARRIER_DETAILS_FILE,
            dtype={"DOT_NUMBER": str}
        )

        carrier_details["DOT_NUMBER"] = (
            carrier_details["DOT_NUMBER"]
            .astype(str)
            .str.strip()
        )
    else:
        # Create empty dataframe if file doesn't exist
        carrier_details = pd.DataFrame()

    existing_dots = set()

    if not carrier_details.empty and "DOT_NUMBER" in carrier_details.columns:
        existing_dots = set(
            carrier_details["DOT_NUMBER"]
            .dropna()
            .astype(str)
            .str.strip()
        )

    # ---------------------------------------------------------
    # 3. Find DOTs missing from carrier_details.csv
    # ---------------------------------------------------------
    missing_dots = sorted(daily_dots - existing_dots)

    print(f"Already in carrier_details.csv: {len(daily_dots & existing_dots)}")
    print(f"Missing carrier details: {len(missing_dots)}")

    # ---------------------------------------------------------
    # 4. Get missing carrier details from FMCSA function
    # ---------------------------------------------------------
    new_carriers = []

    for i, dot_number in enumerate(missing_dots, start=1):

        print(
            f"[{i}/{len(missing_dots)}] "
            f"Getting carrier details for DOT {dot_number}"
        )

        try:
            details = get_carrier_details(dot_number)

            if details is None:
                print(f"  No data returned for DOT {dot_number}")
                continue

            # Make sure DOT_NUMBER exists
            details["DOT_NUMBER"] = str(dot_number)

            new_carriers.append(details)

            print(f"  Success: {details}")

        except Exception as e:
            print(
                f"  ERROR for DOT {dot_number}: {e}"
            )

    # ---------------------------------------------------------
    # 5. Append new carrier records
    # ---------------------------------------------------------
    if new_carriers:

        new_carriers_df = pd.DataFrame(new_carriers)

        # Make DOT_NUMBER consistent
        new_carriers_df["DOT_NUMBER"] = (
            new_carriers_df["DOT_NUMBER"]
            .astype(str)
            .str.strip()
        )

        if carrier_details.empty:
            updated_carriers = new_carriers_df
        else:
            updated_carriers = pd.concat(
                [
                    carrier_details,
                    new_carriers_df
                ],
                ignore_index=True,
                sort=False
            )

        # Remove accidental duplicates
        updated_carriers = (
            updated_carriers
            .drop_duplicates(
                subset=["DOT_NUMBER"],
                keep="last"
            )
        )

        updated_carriers.to_csv(
            CARRIER_DETAILS_FILE,
            index=False
        )

        print(
            f"\nAdded {len(new_carriers_df)} new carriers "
            f"to {CARRIER_DETAILS_FILE}"
        )

    else:
        print("\nNo new carrier details needed.")

        updated_carriers = carrier_details

    return updated_carriers


if __name__ == "__main__":
    carrier_details = update_carrier_details()

    print(
        f"\nTotal carrier records in cache: "
        f"{len(carrier_details)}"
    )

DOT numbers in daily OOS: 810
Already in carrier_details.csv: 1
Missing carrier details: 809
[1/809] Getting carrier details for DOT 100383


  Success: {'dot_number': '100383', 'data': [{'mcs150_date': '20091202 1011', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '100383', 'dun_bradstreet_no': '35993922', 'phy_omc_region': '08', 'safety_inv_terr': 'D', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '0', 'mcs151_mileage': '41073', 'mcs150_update_code_id': '3', 'prior_revoke_flag': 'N', 'phone': '3072348985', 'fax': '3072657330', 'company_officer_1': 'THOMAS LANGFORD', 'business_org_desc': 'CORPORATION', 'truck_units': '1', 'power_units': '5', 'bus_units': '0', 'fleetsize': 'C', 'carship': 'C;S', 'total_intrastate_drivers': '2', 'mcsipstep': '0', 'mcsipdate': '20101211', 'hm_ind': 'Y', 'interstate_within_100_miles': '1', 'intrastate_within_100_miles': '2', 'total_cdl': '2', 'total_drivers': '3', 'classdef': 'PRIVATE PROPERTY', 'legal_name': 'BUTANE POWER & EQUIPMENT COMPANY', 'phy_street': '507 N BEVERLY ST', 'phy_city': 'CASPER', 'phy_country': 'US', 'phy_state': 'WY', 'phy_zip': '82609-1768

  Success: {'dot_number': '101771', 'data': [{'mcs150_date': '20180726 1034', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '101771', 'dun_bradstreet_no': '2428423', 'phy_omc_region': '01', 'safety_inv_terr': 'F', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '330814', 'mcs150_mileage_year': '2017', 'mcs151_mileage': '280702', 'mcs150_update_code_id': '1', 'prior_revoke_flag': 'N', 'phone': '5188531700', 'fax': '5188531706', 'cell_phone': '5182319700', 'company_officer_1': 'MARILYN BUANNO', 'company_officer_2': 'ERIC MACK', 'business_org_desc': 'CORPORATION', 'truck_units': '4', 'power_units': '4', 'bus_units': '0', 'fleetsize': 'C', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '113047', 'total_intrastate_drivers': '0', 'mcsipstep': '57', 'mcsipdate': '20191001', 'hm_ind': 'N', 'interstate_beyond_100_miles': '2', 'total_cdl': '2', 'total_drivers': '2', 'classdef': 'AUTHORIZED FOR HIRE', 'legal_name': 'BUANNO TRANSPORTATION COMPANY INC', 'phy_stree

  Success: {'dot_number': '102180', 'data': [{'mcs150_date': '20021002 0000', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '102180', 'dun_bradstreet_no': '78713542', 'phy_omc_region': '01', 'safety_inv_terr': 'G', 'carrier_operation': 'A', 'business_org_id': '1', 'mcs150_mileage': '0', 'mcs151_mileage': '0', 'mcs150_update_code_id': '3', 'phone': '9146517148', 'business_org_desc': 'INDIVIDUAL', 'truck_units': '1', 'power_units': '1', 'bus_units': '0', 'fleetsize': 'A', 'carship': 'C', 'total_intrastate_drivers': '0', 'mcsipstep': '0', 'mcsipdate': '20090310', 'hm_ind': 'N', 'interstate_beyond_100_miles': '1', 'interstate_within_100_miles': '0', 'intrastate_beyond_100_miles': '0', 'intrastate_within_100_miles': '0', 'total_cdl': '0', 'total_drivers': '1', 'avg_drivers_leased_per_month': '0', 'classdef': 'EXEMPT FOR HIRE', 'legal_name': 'DOLORES ARKEL', 'dba_name': 'GEORGE ARKEL', 'phy_street': '28 ROE ST', 'phy_city': 'FLORIDA', 'phy_country': 'US', 'phy_state': 'NY', 'phy_

  Success: {'dot_number': '102202', 'data': [{'mcs150_date': '20021120 0000', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '102202', 'phy_omc_region': '01', 'safety_inv_terr': 'S', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '610000', 'mcs150_mileage_year': '2001', 'mcs151_mileage': '600000', 'mcs150_update_code_id': '1', 'phone': '9737785828', 'business_org_desc': 'CORPORATION', 'truck_units': '4', 'power_units': '4', 'bus_units': '0', 'fleetsize': 'C', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '124316', 'total_intrastate_drivers': '0', 'mcsipstep': '99', 'mcsipdate': '20141027', 'hm_ind': 'N', 'interstate_beyond_100_miles': '4', 'interstate_within_100_miles': '0', 'intrastate_beyond_100_miles': '0', 'intrastate_within_100_miles': '0', 'total_cdl': '4', 'total_drivers': '4', 'avg_drivers_leased_per_month': '0', 'classdef': 'OTHER-UNKNOWN', 'legal_name': 'ACTIVE EXPRESS COMPANY INC', 'phy_street': '220 EAST HANOVER AVE', 'phy_city': 'MORRIS 

...

  Success: {'dot_number': '93011', 'data': [{'mcs150_date': '20171025 0000', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '93011', 'dun_bradstreet_no': '4371274', 'phy_omc_region': '05', 'safety_inv_terr': 'E', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '400000', 'mcs150_mileage_year': '2016', 'mcs150_update_code_id': '1', 'prior_revoke_flag': 'N', 'phone': '7403732252', 'fax': '7403736359', 'company_officer_1': 'W SCOTT ELLIOT', 'company_officer_2': 'TRENT ELLIOT', 'business_org_desc': 'CORPORATION', 'truck_units': '4', 'power_units': '4', 'bus_units': '0', 'fleetsize': 'C', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '214952', 'docket2prefix': 'MC', 'docket2': '229557', 'total_intrastate_drivers': '0', 'mcsipstep': '0', 'mcsipdate': '20170831', 'hm_ind': 'Y', 'interstate_beyond_100_miles': '9', 'total_cdl': '9', 'total_drivers': '9', 'avg_drivers_leased_per_month': '0', 'classdef': 'PRIVATE PROPERTY;AUTHORIZED FOR HIRE', 'legal_name': 'MARI

  Success: {'dot_number': '95593', 'data': [{'mcs150_date': '20150518 1223', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '95593', 'dun_bradstreet_no': '47871314', 'phy_omc_region': '04', 'safety_inv_terr': 'A', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '60000', 'mcs150_mileage_year': '2015', 'mcs150_update_code_id': '3', 'phone': '2563324510', 'company_officer_1': 'GWEN BROWN', 'business_org_desc': 'CORPORATION', 'truck_units': '2', 'power_units': '2', 'bus_units': '0', 'fleetsize': 'B', 'carship': 'C', 'total_intrastate_drivers': '0', 'mcsipstep': '0', 'mcsipdate': '20150518', 'hm_ind': 'N', 'interstate_beyond_100_miles': '2', 'total_cdl': '0', 'total_drivers': '2', 'avg_drivers_leased_per_month': '0', 'classdef': 'PRIVATE PROPERTY', 'legal_name': 'FRANKLIN HOMES INC', 'phy_street': '10655 HIGHWAY 43', 'phy_city': 'RUSSELLVILLE', 'phy_country': 'US', 'phy_state': 'AL', 'phy_zip': '35653', 'phy_cnty': '059', 'carrier_mailing_street': '10655 HIGHW

  Success: {'dot_number': '95600', 'data': [{'mcs150_date': '20100805 0000', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '95600', 'dun_bradstreet_no': '31622517', 'phy_omc_region': '04', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '187986', 'mcs150_mileage_year': '2006', 'mcs151_mileage': '40074', 'mcs150_update_code_id': '3', 'prior_revoke_flag': 'N', 'phone': '2054865266', 'fax': '2054862157', 'company_officer_1': 'RICKY FULLER', 'company_officer_2': 'RICHARD LEE FULLER', 'business_org_desc': 'CORPORATION', 'truck_units': '2', 'power_units': '2', 'bus_units': '0', 'fleetsize': 'B', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '544708', 'total_intrastate_drivers': '0', 'mcsipstep': '57', 'mcsipdate': '20100315', 'hm_ind': 'N', 'interstate_beyond_100_miles': '1', 'total_cdl': '1', 'total_drivers': '1', 'classdef': 'PRIVATE PROPERTY;AUTHORIZED FOR HIRE', 'legal_name': 'RICHARD FULLER LUMBER SALES INC', 'dba_name': 'FULLER LUMBER SALES INC', 'ph

  Success: {'dot_number': '95947', 'data': [{'mcs150_date': '20260128 1643', 'add_date': '19740601', 'status_code': 'A', 'dot_number': '95947', 'dun_bradstreet_no': '41360173', 'phy_omc_region': '06', 'safety_inv_terr': 'A', 'carrier_operation': 'A', 'business_org_id': '3', 'mcs150_mileage': '490000', 'mcs150_mileage_year': '2025', 'total_cars': '5', 'mcs150_update_code_id': '3', 'prior_revoke_flag': 'N', 'prior_revoke_dot_number': '95947', 'phone': '5058231441', 'fax': '5058281846', 'company_officer_1': 'NOTAH HOWE', 'business_org_desc': 'CORPORATION', 'truck_units': '15', 'power_units': '15', 'bus_units': '0', 'fleetsize': 'G', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '121026', 'total_intrastate_drivers': '4', 'mcsipstep': '0', 'mcsipdate': '20200709', 'hm_ind': 'N', 'interstate_beyond_100_miles': '3', 'interstate_within_100_miles': '0', 'intrastate_beyond_100_miles': '2', 'intrastate_within_100_miles': '2', 'total_cdl': '2', 'total_drivers': '7', 'avg_drivers_leased_per_mon

  Success: {'dot_number': '97434', 'data': [{'mcs150_date': '20090728 0000', 'add_date': '19740601', 'status_code': 'I', 'dot_number': '97434', 'dun_bradstreet_no': '48936157', 'phy_omc_region': '04', 'safety_inv_terr': 'D', 'carrier_operation': 'A', 'business_org_id': '1', 'mcs150_mileage': '178370', 'mcs150_mileage_year': '2000', 'mcs151_mileage': '150000', 'mcs150_update_code_id': '2', 'prior_revoke_flag': 'N', 'phone': '9048642511', 'fax': '9043882939', 'company_officer_1': 'DAVID GUNN', 'business_org_desc': 'INDIVIDUAL', 'truck_units': '4', 'power_units': '4', 'bus_units': '0', 'fleetsize': 'C', 'carship': 'C', 'docket1prefix': 'MC', 'docket1': '213876', 'docket2prefix': 'MC', 'docket2': '129473', 'total_intrastate_drivers': '0', 'mcsipstep': '55', 'mcsipdate': '20060720', 'hm_ind': 'N', 'interstate_beyond_100_miles': '4', 'total_cdl': '0', 'total_drivers': '4', 'classdef': 'AUTHORIZED FOR HIRE', 'legal_name': 'A-1 GUNN MOVING AND STORAGE INC', 'phy_street': '8461 SANTANA COURT', 

In [2]:
import pandas as pd
from sodapy import Socrata
import time
import os

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "./daily/daily_ooo.csv"   # your 400K-row file
OUTPUT_DIR = "ooo_enriched_batches"

BATCH_SIZE = 100
DELAY_SECONDS = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# SOCrata CLIENT
# ============================================================

client = Socrata(
    "data.transportation.gov",
    None
)

DATASET_ID = "az4n-8mr2"


# ============================================================
# FETCH ONE CARRIER
# ============================================================

def get_carrier_details(dot_number: int) -> dict:

    try:

        where_clause = f"dot_number = {dot_number}"

        results = client.get(
            DATASET_ID,
            where=where_clause,
            limit=1
        )

        if not results:

            return {
                "DOT_NUMBER": dot_number,
                "ERROR": f"No carrier found with DOT number {dot_number}",
                "SUCCESS": False
            }

        row = results[0]

        # ----------------------------------------
        # Extract required fields
        # ----------------------------------------

        enriched = {
            "DOT_NUMBER": row.get("dot_number"),
            "LEGAL_NAME": row.get("legal_name"),
            "DBA_NAME": row.get("dba_name"),

            "PHY_STREET": row.get("phy_street"),
            "PHY_CITY": row.get("phy_city"),
            "PHY_STATE": row.get("phy_state"),
            "PHY_ZIP": row.get("phy_zip"),

            "MAILING_STREET": row.get("mailing_street"),
            "MAILING_CITY": row.get("mailing_city"),
            "MAILING_STATE": row.get("mailing_state"),
            "MAILING_ZIP": row.get("mailing_zip"),

            "TELEPHONE": row.get("telephone"),
            "FAX": row.get("fax"),
            "EMAIL": row.get("email_address"),

            "CARRIER_OPERATION": row.get("carrier_operation"),
            "CARRIER_OPERATION_DESC": row.get("carrier_operation_desc"),

            "ENTITY_TYPE": row.get("entity_type"),

            "OPERATING_STATUS": row.get("operating_status"),
            "OPERATING_STATUS_DESC": row.get("operating_status_desc"),

            "USDOT_STATUS": row.get("usdot_status"),

            "MCS150_DATE": row.get("mcs150_date"),

            "DRIVER_TOTAL": row.get("driver_total"),
            "VEHICLE_TOTAL": row.get("vehicle_total"),

            "OUT_OF_SERVICE_DATE": row.get("oos_date"),

            "SAFETY_RATING": row.get("safety_rating"),

            "SUCCESS": True
        }

        return enriched

    except Exception as e:

        return {
            "DOT_NUMBER": dot_number,
            "ERROR": str(e),
            "SUCCESS": False
        }


# ============================================================
# READ 400K INPUT FILE
# ============================================================

print("Reading input file...")

df = pd.read_csv(
    INPUT_FILE,
    usecols=["DOT_NUMBER"]
)

print(f"Total input rows: {len(df):,}")


# ============================================================
# CLEAN DOT NUMBERS
# ============================================================

df["DOT_NUMBER"] = pd.to_numeric(
    df["DOT_NUMBER"],
    errors="coerce"
)

df = df.dropna(
    subset=["DOT_NUMBER"]
)

df["DOT_NUMBER"] = df["DOT_NUMBER"].astype("int64")


# ============================================================
# FIND UNIQUE DOT NUMBERS
#
# drop_duplicates() preserves the order in which
# the DOT numbers first appear in the input file.
# ============================================================

unique_dots = (
    df["DOT_NUMBER"]
    .drop_duplicates()
    .tolist()
)

total_unique = len(unique_dots)

print(f"Unique DOT numbers: {total_unique:,}")
print(
    f"Duplicates removed: "
    f"{len(df) - total_unique:,}"
)

print("=" * 70)


# ============================================================
# FETCH UNIQUE CARRIERS SEQUENTIALLY
# ============================================================

enriched_rows = []

for index, dot in enumerate(unique_dots, start=1):

    print(
        f"[{index:,}/{total_unique:,}] "
        f"Fetching DOT {dot}"
    )

    result = get_carrier_details(dot)

    enriched_rows.append(result)

    if result.get("SUCCESS"):

        print(
            f"    ✓ "
            f"{result.get('LEGAL_NAME', '')}"
        )

    else:

        print(
            f"    ✗ "
            f"{result.get('ERROR')}"
        )

    # ----------------------------------------
    # Save every 100 unique carriers
    # ----------------------------------------

    if (
        len(enriched_rows) == BATCH_SIZE
        or index == total_unique
    ):

        batch_start = index - len(enriched_rows) + 1
        batch_end = index

        batch_df = pd.DataFrame(enriched_rows)

        output_file = os.path.join(
            OUTPUT_DIR,
            f"carriers_{batch_start:06d}_{batch_end:06d}.csv"
        )

        batch_df.to_csv(
            output_file,
            index=False
        )

        print()
        print(
            f"    >>> Saved batch: "
            f"{output_file}"
        )
        print()

        # Clear memory before next batch
        enriched_rows = []

    # ----------------------------------------
    # Delay between API requests
    # ----------------------------------------

    if index < total_unique:
        time.sleep(DELAY_SECONDS)


# ============================================================
# COMPLETE
# ============================================================

print("=" * 70)
print("DONE")
print(f"Input rows:       {len(df):,}")
print(f"Unique DOTs:      {total_unique:,}")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 70)

Reading input file...
Total input rows: 1,000
Unique DOT numbers: 810
Duplicates removed: 190
[1/810] Fetching DOT 1438
    ✓ AUSTIN URETHANE INC
[2/810] Fetching DOT 6050
    ✓ S M TRANSPORT INC
[3/810] Fetching DOT 7660
    ✓ NEW CHURCH FARMERS SUPPLY INC
[4/810] Fetching DOT 11891
    ✓ H DAVID PITZER TRUCKING INC
[5/810] Fetching DOT 13172
    ✓ LARRY TRAPP TRUCKING INC
[6/810] Fetching DOT 14662
    ✓ ALFRED DANIELS INC
[7/810] Fetching DOT 14986
    ✓ STOCKTON OIL CO
[8/810] Fetching DOT 15330
    ✓ THRUWAY MESSENGER SERVICE INC
[9/810] Fetching DOT 15674
    ✓ LOREN OBRIST EXCAVATING INC
[10/810] Fetching DOT 16535
    ✓ EMPIRE WAREHOUSING & LEASING COMPANY INC
[11/810] Fetching DOT 19279
    ✓ NORTH HAVEN TRANSPORTATION CO INC
[12/810] Fetching DOT 22943
    ✓ A S N GRAIN INC
[13/810] Fetching DOT 23285
    ✓ M PAGANO & SONS INC
[14/810] Fetching DOT 23562
    ✓ PRYSLAK TRANSPORTATION INC
[15/810] Fetching DOT 26171
    ✓ SILVER LINE INC
[16/810] Fetching DOT 26479
    ✓ FARR'S